# Dataset SMILES Counts

Counts unique SMILES across all downloaded pre-training datasets.
Includes RDKit validity checks and cross-dataset overlap.

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import OrderedDict
from tqdm.auto import tqdm

try:
    from rdkit import Chem
    HAS_RDKIT = True
except ImportError:
    HAS_RDKIT = False
    print('RDKit not available -- skipping validity checks')

DATA_DIR = Path('..')

def load_smiles(path, smiles_col, name):
    """Load SMILES from a CSV/parquet file."""
    path = Path(path)
    if not path.exists():
        print(f'  {name}: NOT FOUND at {path}')
        return None
    if path.suffix == '.parquet':
        df = pd.read_parquet(path, columns=[smiles_col])
    elif path.suffix == '.gz':
        df = pd.read_csv(path, usecols=[smiles_col])
    else:
        df = pd.read_csv(path, usecols=[smiles_col])
    smiles = df[smiles_col].dropna().tolist()
    print(f'  {name}: {len(smiles):>10,} rows loaded')
    return smiles

## 1. Load all datasets

In [ ]:
SMALL = DATA_DIR / 'data/graphium/neurips2023/small-dataset'
LARGE = DATA_DIR / 'data/graphium/neurips2023/large-dataset'

all_datasets = OrderedDict()

# --- ToyMix ---
print('ToyMix:')
all_datasets['QM9'] = load_smiles(SMALL / 'qm9.csv', 'smiles', 'QM9')
all_datasets['Tox21'] = load_smiles(SMALL / 'Tox21-7k-12-labels.csv', 'smiles', 'Tox21')
all_datasets['ZINC12k'] = load_smiles(SMALL / 'ZINC12k.csv', 'smiles', 'ZINC12k')

# # --- LargeMix ---
# print('\nLargeMix:')
# all_datasets['L1000 VCAP'] = load_smiles(LARGE / 'LINCS_L1000_VCAP_0-2_th2.csv.gz', 'SMILES', 'L1000 VCAP')
# all_datasets['L1000 MCF7'] = load_smiles(LARGE / 'LINCS_L1000_MCF7_0-2_th2.csv.gz', 'SMILES', 'L1000 MCF7')
# all_datasets['PCBA'] = load_smiles(LARGE / 'PCBA_1328_1564k.parquet', 'SMILES', 'PCBA')
# all_datasets['PCQM4M'] = load_smiles(LARGE / 'PCQM4M_G25_N4.parquet', 'ordered_smiles', 'PCQM4M')

# # --- Cell morphology embeddings ---
# print('\nCell Morphology:')
# all_datasets['RxRx3'] = load_smiles(DATA_DIR / 'data/rxrx3/rxrx3_smiles_embeddings.csv', 'SMILES_nometa', 'RxRx3')
# all_datasets['BBBC047'] = load_smiles(Path('../../data/bbbc047/bbbc047_smiles_embeddings.csv'), 'SMILES', 'BBBC047')

# # --- DTI ---
# print('\nDTI:')
# all_datasets['DTI (ESM2)'] = load_smiles(DATA_DIR / 'data/dti-processed/dti_esm2_100k.csv', 'SMILES_nometa', 'DTI (ESM2)')
# all_datasets['DTI (ESM-C)'] = load_smiles(DATA_DIR / 'data/dti-processed/dti_esmc_100k.csv', 'SMILES_nometa', 'DTI (ESM-C)')

# --- LPM-24 ---
print('\nLPM-24:')
lpm24_raw = DATA_DIR / 'data/lpm24/lpm24_raw.parquet'
all_datasets['LPM-24'] = load_smiles(lpm24_raw, 'molecule', 'LPM-24') if lpm24_raw.exists() else None
if all_datasets['LPM-24'] is None:
    # Try the processed version
    all_datasets['LPM-24'] = load_smiles(DATA_DIR / 'data/dti-processed/lpm24_pubmedbert.csv', 'SMILES_nometa', 'LPM-24 (processed)')

# Remove datasets that weren't found
all_datasets = {k: v for k, v in all_datasets.items() if v is not None}
print(f'\nLoaded {len(all_datasets)} datasets.')

## 2. Unique SMILES count + RDKit validity

In [8]:
rows = []
for name, smiles_list in tqdm(all_datasets.items(), desc="Counting SMILES"):
    total = len(smiles_list)
    str_unique = len(set(smiles_list))

    # RDKit canonicalization + validity
    if HAS_RDKIT:
        canonical = {}  # raw SMILES -> canonical SMILES (or None if invalid)
        for s in tqdm(smiles_list, desc=f"  RDKit {name}", leave=False):
            if s not in canonical:
                mol = Chem.MolFromSmiles(s)
                canonical[s] = Chem.MolToSmiles(mol) if mol is not None else None

        n_invalid_str = sum(1 for v in canonical.values() if v is None)
        valid_canonical = {v for v in canonical.values() if v is not None}
        n_canon_unique = len(valid_canonical)
        n_valid_str = str_unique - n_invalid_str

        # Duplicates: same canonical SMILES from different raw strings
        canon_dup = n_valid_str - n_canon_unique  # extra unique strings that collapse after canonicalization

        rows.append({
            'Dataset': name,
            'Total Rows': f'{total:,}',
            'Unique (string)': f'{str_unique:,}',
            'Row Duplicates': f'{total - str_unique:,}',
            'RDKit Valid': f'{n_valid_str:,} / {str_unique:,}',
            'Unique (canonical)': f'{n_canon_unique:,}',
            'Canon. Collapsed': f'{canon_dup:,}',
            'RDKit Invalid': f'{n_invalid_str:,}',
        })
    else:
        rows.append({
            'Dataset': name,
            'Total Rows': f'{total:,}',
            'Unique (string)': f'{str_unique:,}',
            'Row Duplicates': f'{total - str_unique:,}',
            'RDKit Valid': 'N/A',
            'Unique (canonical)': 'N/A',
            'Canon. Collapsed': 'N/A',
            'RDKit Invalid': 'N/A',
        })

count_df = pd.DataFrame(rows).set_index('Dataset')
count_df.style.set_caption('Unique SMILES and RDKit Validity per Dataset')

Counting SMILES:   0%|          | 0/1 [00:00<?, ?it/s]

  RDKit LPM-24:   0%|          | 0/160560 [00:00<?, ?it/s]

[17:25:32] WARNING: not removing hydrogen atom without neighbors
[17:25:34] WARNING: not removing hydrogen atom without neighbors
[17:25:38] WARNING: not removing hydrogen atom without neighbors
[17:25:38] WARNING: not removing hydrogen atom without neighbors
[17:25:39] WARNING: not removing hydrogen atom without neighbors
[17:25:45] WARNING: not removing hydrogen atom without neighbors
[17:25:55] WARNING: not removing hydrogen atom without neighbors
[17:26:04] WARNING: not removing hydrogen atom without neighbors
[17:26:07] WARNING: not removing hydrogen atom without neighbors
[17:26:11] WARNING: not removing hydrogen atom without neighbors
[17:26:14] WARNING: not removing hydrogen atom without neighbors
[17:26:24] WARNING: not removing hydrogen atom without neighbors
[17:26:28] WARNING: not removing hydrogen atom without neighbors
[17:26:28] WARNING: not removing hydrogen atom without neighbors
[17:26:28] WARNING: not removing hydrogen atom without neighbors
[17:26:28] WARNING: not r

,Total Rows,Unique (string),Row Duplicates,RDKit Valid,Unique (canonical),Canon. Collapsed,RDKit Invalid
Dataset,,,,,,,
LPM-24,"160,560","160,560",0,"160,560 / 160,560","160,560",0,0


## 3. Composite dataset counts

Unique SMILES in the combined pre-training mixes.

In [9]:
def canonicalize(smiles_list):
    """Return set of canonical SMILES (skipping invalid)."""
    canon = set()
    for s in smiles_list:
        mol = Chem.MolFromSmiles(s)
        if mol is not None:
            canon.add(Chem.MolToSmiles(mol))
    return canon

def unique_smiles_set(names, use_canonical=True):
    """Union of SMILES across named datasets."""
    combined = set()
    for n in names:
        if n in all_datasets:
            if use_canonical and HAS_RDKIT:
                combined.update(canonicalize(all_datasets[n]))
            else:
                combined.update(all_datasets[n])
    return combined

combos = OrderedDict()
combos['ToyMix'] = ['QM9', 'Tox21', 'ZINC12k']
combos['LargeMix'] = ['L1000 VCAP', 'L1000 MCF7', 'PCBA', 'PCQM4M']
combos['ToyMix + DTI'] = ['QM9', 'Tox21', 'ZINC12k', 'DTI (ESM2)']
combos['ToyMix + RxRx3'] = ['QM9', 'Tox21', 'ZINC12k', 'RxRx3']
combos['ToyMix + BBBC047'] = ['QM9', 'Tox21', 'ZINC12k', 'BBBC047']
combos['ToyMix + LPM-24'] = ['QM9', 'Tox21', 'ZINC12k', 'LPM-24']
combos['ToyMix + DTI + RxRx3'] = ['QM9', 'Tox21', 'ZINC12k', 'DTI (ESM2)', 'RxRx3']
combos['LargeMix + DTI'] = ['L1000 VCAP', 'L1000 MCF7', 'PCBA', 'PCQM4M', 'DTI (ESM2)']
combos['All'] = list(all_datasets.keys())

combo_rows = []
for label, names in combos.items():
    available = [n for n in names if n in all_datasets]
    if not available:
        continue
    s_canon = unique_smiles_set(available, use_canonical=True)
    s_raw = unique_smiles_set(available, use_canonical=False)
    combo_rows.append({
        'Combination': label,
        'Datasets': ', '.join(available),
        'Unique (string)': f'{len(s_raw):,}',
        'Unique (canonical)': f'{len(s_canon):,}',
    })

combo_df = pd.DataFrame(combo_rows).set_index('Combination')
combo_df.style.set_caption('Unique SMILES in Combined Pre-training Mixes')

[17:26:33] WARNING: not removing hydrogen atom without neighbors
[17:26:35] WARNING: not removing hydrogen atom without neighbors
[17:26:39] WARNING: not removing hydrogen atom without neighbors
[17:26:39] WARNING: not removing hydrogen atom without neighbors
[17:26:39] WARNING: not removing hydrogen atom without neighbors
[17:26:45] WARNING: not removing hydrogen atom without neighbors
[17:26:54] WARNING: not removing hydrogen atom without neighbors
[17:27:04] WARNING: not removing hydrogen atom without neighbors
[17:27:06] WARNING: not removing hydrogen atom without neighbors
[17:27:10] WARNING: not removing hydrogen atom without neighbors
[17:27:12] WARNING: not removing hydrogen atom without neighbors
[17:27:22] WARNING: not removing hydrogen atom without neighbors
[17:27:26] WARNING: not removing hydrogen atom without neighbors
[17:27:26] WARNING: not removing hydrogen atom without neighbors
[17:27:26] WARNING: not removing hydrogen atom without neighbors
[17:27:26] WARNING: not r

,Datasets,Unique (string),Unique (canonical)
Combination,,,
ToyMix + LPM-24,LPM-24,"160,560","160,560"
All,LPM-24,"160,560","160,560"


## 4. Cross-dataset overlap (canonical SMILES)

Pairwise overlap between datasets using RDKit-canonicalized SMILES.

In [ ]:
# Build canonical SMILES sets per dataset
canon_sets = {}
for name, smiles_list in tqdm(all_datasets.items(), desc="Canonicalizing"):
    canon_sets[name] = canonicalize(smiles_list)
    print(f'  {name}: {len(canon_sets[name]):,} unique canonical SMILES')

# Pairwise overlap
ds_names = list(canon_sets.keys())
print(f'\n{"Dataset A":<20} {"Dataset B":<20} {"A size":>10} {"B size":>10} {"Overlap":>10} {"A⊂B %":>8} {"B⊂A %":>8}')
print('-' * 90)
for i, a in enumerate(ds_names):
    for b in ds_names[i+1:]:
        overlap = canon_sets[a] & canon_sets[b]
        pct_a_in_b = 100 * len(overlap) / len(canon_sets[a]) if canon_sets[a] else 0
        pct_b_in_a = 100 * len(overlap) / len(canon_sets[b]) if canon_sets[b] else 0
        print(f'{a:<20} {b:<20} {len(canon_sets[a]):>10,} {len(canon_sets[b]):>10,} {len(overlap):>10,} {pct_a_in_b:>7.1f}% {pct_b_in_a:>7.1f}%')

# Special focus: ToyMix in LPM-24
toymix_names = ['QM9', 'Tox21', 'ZINC12k']
toymix_available = [n for n in toymix_names if n in canon_sets]
if toymix_available and 'LPM-24' in canon_sets:
    toymix_all = set()
    for n in toymix_available:
        toymix_all |= canon_sets[n]
    lpm = canon_sets['LPM-24']
    overlap = toymix_all & lpm
    only_toymix = toymix_all - lpm
    print(f'\n--- ToyMix vs LPM-24 ---')
    print(f'ToyMix unique canonical:  {len(toymix_all):,}')
    print(f'LPM-24 unique canonical:  {len(lpm):,}')
    print(f'Overlap:                  {len(overlap):,} ({100*len(overlap)/len(toymix_all):.1f}% of ToyMix)')
    print(f'ToyMix only (not in LPM): {len(only_toymix):,}')
    if only_toymix:
        print(f'\nSample ToyMix SMILES NOT in LPM-24 (first 10):')
        for s in sorted(only_toymix)[:10]:
            print(f'  {s}')